In [1]:
import pandas as pd 
import numpy as np 

service = pd.read_csv(r'/Users/yoganshusharma/Desktop/xiaomi_report_engine/mi_smart_report (6).csv')
service.columns

Index(['Policy ID', 'SOLD DATE', 'PLAN TYPE', 'ASC Code', 'ASC Name',
       'ASP Code', 'ASP Name', 'Customer city', 'Customer state',
       'CUSTOMER PRICE', 'ASC PRICE', 'PAYMENT STATUS', 'DEVICE MODEL',
       'SKU ID', 'POLICY START DATE', 'POLICY END DATE', 'CUST NAME',
       'CUST CONTACT', 'Customer email', 'Cust address', 'IMEI/SERIAL NUMBER',
       'MID'],
      dtype='str')

In [2]:
service.shape

(953, 22)

In [3]:
service = service[service['PAYMENT STATUS']== True]

In [4]:
service.shape

(854, 22)

In [5]:
master = pd.read_excel(
    r"current_service_master.xlsx"
)

In [6]:
master.columns

Index(['Agency_Code', 'Agency_Name', 'Agency_Operated_by',
       'Service_Centre_Type', 'Level_of_Repair', 'Type', 'Company_Name', 'RSM',
       'RSM_Contact_No', 'RSM_Email_ID', 'Filed-service_manager',
       'FSM_Contact_No', 'FSM_Email_ID', 'Region', 'City', 'Tier', 'State',
       'State-2', 'Subtitle(Landmark)', 'Address', 'PIN-Code',
       'Service_center_Tel', 'ASC_Mail_ID', 'Hours_of_Operation', 'Status_',
       'Closed_date_if_Applicable', 'NSM Name', 'District'],
      dtype='str')

In [7]:
df_merged = service.merge(
    master,
    left_on='ASC Code',
    right_on='Agency_Code',
    how='left'
)

In [8]:
df_merged.shape

(854, 50)

In [9]:
pivot_df = pd.pivot_table(
    df_merged,
    index=['Region', 'State-2', 'Agency_Name'],
    values='CUSTOMER PRICE',
    aggfunc=['count', 'sum'],
    margins=True,
    margins_name='Total'
)

pivot_df.columns = [
    'Count of PAYMENT STATUS',
    'Sum of CUSTOMER PRICE'
]

pivot_df.to_excel('zonal_report.xlsx')

In [ ]:
service['ASC Code'] = (
    service['ASC Code']
    .astype(str)
    .str.strip()
)

master['Agency_Code'] = (
    master['Agency_Code']
    .astype(str)
    .str.strip()
)

# =========================
# FILTER PAYMENT STATUS = TRUE
# =========================

service = service[
    service['PAYMENT STATUS']
        .astype(str)
        .str.strip()
        .str.upper()
        .isin(['TRUE'])
]

# =========================
# MERGE MASTER
# =========================

df_merged = service.merge(
    master,
    left_on='ASC Code',
    right_on='Agency_Code',
    how='left'
)

# =========================
# FILL NULLS AS BLANK
# =========================

df_merged['region'] = df_merged['region'].fillna('Blank')
df_merged['State'] = df_merged['State'].fillna('Blank')
df_merged['ASC_Name_BI'] = df_merged['ASC_Name_BI'].fillna('Blank')

report_df = (
    df_merged.groupby(
        ['region', 'State', 'ASC_Name_BI'],
        as_index=False
    )
    .agg(
        Unit=('PAYMENT STATUS', 'count'),
        GWP=('CUSTOMER PRICE', 'sum')
    )
)

regions = [z for z in report_df['region'].unique() if z != 'Blank']

if 'Blank' in report_df['region'].unique():
    regions.append('Blank')

final_rows = []

grand_unit_total = 0
grand_gwp_total = 0

for region in regions:

    region_df = report_df[report_df['region'] == region]

    region_unit_total = 0
    region_gwp_total = 0

    first_region_row = True

    states = [s for s in region_df['State'].unique() if s != 'Blank']

    if 'Blank' in region_df['State'].unique():
        states.append('Blank')

    for state in states:

        state_df = region_df[region_df['State'] == state]

        state_unit_total = state_df['Unit'].sum()
        state_gwp_total = state_df['GWP'].sum()

        region_unit_total += state_unit_total
        region_gwp_total += state_gwp_total

        grand_unit_total += state_unit_total
        grand_gwp_total += state_gwp_total

        first_state_row = True

        for _, row in state_df.iterrows():

            final_rows.append({
                'region': region if first_region_row else '',
                'State': state if first_state_row else '',
                'Service Center Name': row['ASC_Name_BI'],
                'Unit': int(row['Unit']),
                'GWP': int(row['GWP'])
            })

            first_region_row = False
            first_state_row = False


        final_rows.append({
            'region': '',
            'State': f'{state} Total',
            'Service Center Name': '',
            'Unit': int(state_unit_total),
            'GWP': int(state_gwp_total)
        })


    final_rows.append({
        'region': f'{region} Total',
        'State': '',
        'Service Center Name': '',
        'Unit': int(region_unit_total),
        'GWP': int(region_gwp_total)
    })

final_rows.append({
    'region': 'Grand Total',
    'State': '',
    'Service Center Name': '',
    'Unit': int(grand_unit_total),
    'GWP': int(grand_gwp_total)
})

final_report = pd.DataFrame(final_rows)


final_report.to_excel(
    'final_report.xlsx',
    index=False
)

print("Zonal report generated successfully.")

KeyError: 'ASC_Code'